## Harmony Batch Correction

First, import the necessary libraries, `scanpy` and `harmonypy`, and includes a fix for potential Windows DLL loading issues with PyTorch-related dependencies.

In [1]:
import ctypes
import os
import platform
from importlib.util import find_spec

# Force-load c10.dll before other imports to prevent the WinError 1114 crash
if platform.system() == "Windows":
    try:
        if (spec := find_spec("torch")) and spec.origin:
            torch_lib = os.path.join(os.path.dirname(spec.origin), "lib")
            c10_path = os.path.join(torch_lib, "c10.dll")
            if os.path.exists(c10_path):
                # Add directory to DLL search path for dependencies
                os.add_dll_directory(torch_lib)
                # Load the DLL directly
                ctypes.CDLL(os.path.normpath(c10_path))
    except Exception:
        pass

# Now import your standard libraries securely
import scanpy as sc
import harmonypy as hm

print("Libraries imported successfully without DLL error!")

Libraries imported successfully without DLL error!


This cell loads the pre-processed AnnData object containing PCA results from a `.h5ad` file.

In [10]:
#read real data
adata = sc.read_h5ad("adata_after_pca.h5ad")

Extract the PCA embeddings and the 'batch' metadata, which are required for running Harmony batch correction.

In [11]:
# 2. Extract PCA embeddings and metadata
pca_data = adata.obsm["X_pca"]
meta_data = adata.obs[["batch"]]  # Ensure 'batch' matches your column name

Then, perform batch correction using Harmony on the extracted PCA data, considering 'batch' as the variable to correct for.

In [12]:
# 3. Run Harmony directly (Bypasses Scanpy's wrapper and its PyTorch/DLL check)
harmony_out = hm.run_harmony(pca_data, meta_data, vars_use=["batch"])

2026-05-02 17:04:59,164 - harmonypy - INFO - Running Harmony (PyTorch on cpu)
2026-05-02 17:04:59,167 - harmonypy - INFO -   Parameters:
2026-05-02 17:04:59,168 - harmonypy - INFO -     max_iter_harmony: 10
2026-05-02 17:04:59,170 - harmonypy - INFO -     max_iter_kmeans: 20
2026-05-02 17:04:59,171 - harmonypy - INFO -     epsilon_cluster: 1e-05
2026-05-02 17:04:59,173 - harmonypy - INFO -     epsilon_harmony: 0.0001
2026-05-02 17:04:59,174 - harmonypy - INFO -     nclust: 100
2026-05-02 17:04:59,176 - harmonypy - INFO -     block_size: 0.05
2026-05-02 17:04:59,178 - harmonypy - INFO -     lamb: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
2026-05-02 17:04:59,181 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-05-02 17:04:59,182 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-05-02 17:04:59,187 - harmonypy - INFO -     verbose: True
2026-05-02 17:04:59,189 - harmonypy - INFO -     random_state: 0
2026-05-02 17:04:59,190 - harmonypy - INFO -   Data: 50 

2026-05-02 17:04:59,356 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-05-02 17:05:03,117 - harmonypy - INFO - KMeans initialization complete.
2026-05-02 17:05:04,180 - harmonypy - INFO - Iteration 1 of 10
2026-05-02 17:05:24,077 - harmonypy - INFO - Iteration 2 of 10
2026-05-02 17:05:38,259 - harmonypy - INFO - Iteration 3 of 10
2026-05-02 17:05:50,215 - harmonypy - INFO - Iteration 4 of 10
2026-05-02 17:05:58,924 - harmonypy - INFO - Iteration 5 of 10
2026-05-02 17:06:08,431 - harmonypy - INFO - Iteration 6 of 10
2026-05-02 17:06:17,103 - harmonypy - INFO - Iteration 7 of 10
2026-05-02 17:06:26,754 - harmonypy - INFO - Iteration 8 of 10
2026-05-02 17:06:35,338 - harmonypy - INFO - Converged after 8 iterations


It stores the batch-corrected PCA embeddings, resulting from the Harmony run, back into the AnnData object under 'X_pca_harmony'.

In [13]:
# 4. Store corrected embeddings back into AnnData
adata.obsm["X_pca_harmony"] = harmony_out.Z_corr

Verify the shapes of the original and Harmony-corrected PCA embeddings to ensure consistency.

In [14]:
print(f"Original PCA shape: {adata.obsm['X_pca'].shape}")
print(f"Harmony PCA shape: {adata.obsm['X_pca_harmony'].shape}")

Original PCA shape: (69339, 50)
Harmony PCA shape: (69339, 50)


Save the AnnData object, including the Harmony batch-corrected embeddings, to a new `.h5ad` file for future use.

In [15]:
adata.write_h5ad("combined_harmony_corrected.h5ad")
print("Saved data successfully. Ready for downstream steps later!")

Saved data successfully. Ready for downstream steps later!
